# 1 - Separando fatos e dimensão.



In [0]:
from pyspark.sql import functions as F

spark.sql("CREATE SCHEMA IF NOT EXISTS portfolio_energia.gold")

df_regional = spark.table("portfolio_energia.silver.balanco_energia")
df_pld      = spark.table("portfolio_energia.silver.pld_horario")
df_sin      = spark.table("portfolio_energia.silver.balanco_energia_sin")

# criando a dimensão de submercado
dim_submercado = df_regional.select("id_subsistema", "nom_subsistema").distinct()
dim_submercado.write.format("delta").mode("overwrite") \
    .saveAsTable("portfolio_energia.gold.dim_submercado")

# criando a dimensão de tempo (dias da semana, trimestre, dia, hora e mes separados)
instantes = (
    df_regional.select("instante")
    .union(df_pld.select("instante"))
    .union(df_sin.select("instante"))
    .distinct()
)

dim_tempo = (
    instantes
    .withColumn("data", F.to_date("instante"))
    .withColumn("ano", F.year("instante"))
    .withColumn("mes", F.month("instante"))
    .withColumn("dia", F.dayofmonth("instante"))
    .withColumn("hora", F.hour("instante"))
    .withColumn("trimestre", F.quarter("instante"))
    .withColumn("dia_semana", F.date_format("instante", "EEEE"))
)
dim_tempo.write.format("delta").mode("overwrite") \
    .saveAsTable("portfolio_energia.gold.dim_tempo")

# criando a tabela fato da geração em nível Regional (submercados)
fact_geracao_carga = df_regional  # já contém id_subsistema e instante
fact_geracao_carga.write.format("delta").mode("overwrite") \
    .saveAsTable("portfolio_energia.gold.fact_geracao_carga")

# criando a tabela fato do PLD
fact_pld = df_pld  # já contém id_subsistema e instante
fact_pld.write.format("delta").mode("overwrite") \
    .saveAsTable("portfolio_energia.gold.fact_pld")

# criando a tabela fato da geração em nível Nacional
fact_geracao_carga_sin = df_sin  # já contém instante
fact_geracao_carga_sin.write.format("delta").mode("overwrite") \
    .saveAsTable("portfolio_energia.gold.fact_geracao_carga_sin")

# 2 - Construindo as tabelas de métrica

### 2.1 - Criando a tabela agg_intercambio_preco: tabela de métrica que responde à primeira parte da pergunta: como o intercâmbio afeta o preço. Compara o PLD de cada subsistema com o do SE (referência do mercado) e separa os meses em horas em que o subsistema exportou e horas em que importou

In [0]:
%sql
CREATE OR REPLACE TABLE portfolio_energia.gold.agg_intercambio_preco AS
WITH pld_se AS (
  SELECT
    instante,
    pld_hora AS pld_se
  FROM portfolio_energia.gold.fact_pld
  WHERE id_subsistema = 'SE'
),
base AS (
  SELECT
    p.id_subsistema,
    t.ano,
    t.mes,
    CASE
      WHEN b.val_intercambio > 0 THEN 'EXPORTADOR'
      ELSE 'IMPORTADOR'
    END                              AS papel_intercambio,
    b.val_intercambio,
    p.pld_hora,
    p.pld_hora - s.pld_se            AS spread_se
  FROM portfolio_energia.gold.fact_pld p
  JOIN pld_se s
    ON p.instante = s.instante
  JOIN portfolio_energia.gold.fact_geracao_carga b
    ON  p.instante      = b.instante
    AND p.id_subsistema = b.id_subsistema
  JOIN portfolio_energia.gold.dim_tempo t
    ON p.instante = t.instante
  WHERE p.id_subsistema <> 'SE'
    AND b.val_intercambio <> 0
)
SELECT
  id_subsistema,
  ano,
  mes,
  MAKE_DATE(ano, mes, 1)                                              AS mes_referencia,
  papel_intercambio,
  COUNT(*)                                                            AS qtd_horas,
  ROUND(AVG(val_intercambio), 2)                                      AS intercambio_medio_mwmed,
  ROUND(AVG(pld_hora), 2)                                             AS pld_medio,
  ROUND(AVG(spread_se), 2)                                            AS spread_medio_se,
  ROUND(100.0 * SUM(CASE WHEN ABS(spread_se) > 0.01 THEN 1 ELSE 0 END)
        / COUNT(*), 2)                                                AS pct_horas_descoladas
FROM base
GROUP BY id_subsistema, ano, mes, papel_intercambio;

SELECT *
FROM portfolio_energia.gold.agg_intercambio_preco

### 2.2 - Criando a tabela agg_impacto_fonte: tabela de métrica que responde à segunda parte da pergunta: qual tipo de usina mais se associa às variações do PLD. Para cada fonte, compara o preço nas horas em que ela gerou acima do normal com o preço nas horas em que gerou abaixo do normal.

In [0]:
%sql
CREATE OR REPLACE TABLE portfolio_energia.gold.agg_impacto_fonte AS
WITH pld_balanco AS (
  SELECT
    p.id_subsistema,
    t.ano,
    t.mes,
    t.hora,
    CAST(p.pld_hora AS DOUBLE) AS pld_hora,
    CAST(p.pld_hora AS DOUBLE)
      - AVG(CAST(p.pld_hora AS DOUBLE)) OVER (
          PARTITION BY p.id_subsistema, t.ano, t.mes, t.hora
        )                      AS desvio_pld,
    CAST(b.val_gerhidraulica AS DOUBLE) AS ger_hidraulica,
    CAST(b.val_gertermica    AS DOUBLE) AS ger_termica,
    CAST(b.val_gereolica     AS DOUBLE) AS ger_eolica,
    CAST(b.val_gersolar      AS DOUBLE) AS ger_solar
  FROM portfolio_energia.gold.fact_pld p
  JOIN portfolio_energia.gold.fact_geracao_carga b
    ON  p.instante      = b.instante
    AND p.id_subsistema = b.id_subsistema
  JOIN portfolio_energia.gold.dim_tempo t
    ON p.instante = t.instante
),
base AS (
  SELECT
    pb.id_subsistema,
    pb.ano,
    pb.mes,
    pb.hora,
    pb.pld_hora,
    pb.desvio_pld,
    s.fonte,
    s.geracao_mwmed
  FROM pld_balanco pb
  LATERAL VIEW STACK(
    4,
    'HIDRAULICA', pb.ger_hidraulica,
    'TERMICA',    pb.ger_termica,
    'EOLICA',     pb.ger_eolica,
    'SOLAR',      pb.ger_solar
  ) s AS fonte, geracao_mwmed
),
com_media AS (
  SELECT
    *,
    AVG(geracao_mwmed) OVER (
      PARTITION BY fonte, id_subsistema, ano, mes, hora
    ) AS media_ref_mwmed
  FROM base
),
classificado AS (
  SELECT
    fonte,
    id_subsistema,
    ano,
    pld_hora,
    desvio_pld,
    CASE
      WHEN geracao_mwmed > media_ref_mwmed THEN 'ALTA'
      ELSE 'BAIXA'
    END AS faixa_geracao
  FROM com_media
  WHERE media_ref_mwmed >= 100
)
SELECT
  fonte,
  id_subsistema,
  ano,
  COUNT(*)                                                               AS qtd_horas,
  ROUND(AVG(pld_hora), 2)                                                AS pld_medio_geral,
  ROUND(AVG(CASE WHEN faixa_geracao = 'ALTA'  THEN desvio_pld END), 2)   AS desvio_medio_alta,
  ROUND(AVG(CASE WHEN faixa_geracao = 'BAIXA' THEN desvio_pld END), 2)   AS desvio_medio_baixa,
  ROUND(
    AVG(CASE WHEN faixa_geracao = 'ALTA'  THEN desvio_pld END)
  - AVG(CASE WHEN faixa_geracao = 'BAIXA' THEN desvio_pld END), 2)       AS impacto
FROM classificado
GROUP BY fonte, id_subsistema, ano;

SELECT *
FROM portfolio_energia.gold.agg_impacto_fonte